### Consigna del TP 1C

- Por qué es necesario refrescar el servidor_con_interfaz.py para poder ver los mensajes?

  Es necesario refrescar cada unos segundos dado que la comunicacion funciona sobre el protocolo HTTP. La forma que tiene este protocolo de comunicarse es a traves de un REQUEST/RESPONSE, como no hay actualizacion automiatica, el cliente solo recibe informacion cuando la solicita explicitamente al servidor.
    
- Funciona apagar servidor ? Cómo funciona ? Qué hace ? \
  Apagar servidor funciona. Lo que hace es establecer el flag de la variable "apagado" en True. Apagado es del tipo threading.Event, un objeto que permite sincronizar procesos usando un flag booleano. Una vez que apagado queda en True, los procesos hijos que estan conectados a los clientes van a salir de su ciclo while con condicion : "while not apagado.is_set()". Este while esta dentro de una estructura Try que  agarra errores en caso de que los haya, y que finalmente cierra la conexion con los clientes. La pagina web se sigue viendo pq el servidor Flask es ajeno a nuestras conexiones.
- Escribir una aplicación cliente servidor en python que:
  - Envíe un archivo pequeño.
  - Que muestre las direcciones y puertos de todos los clientes conectados del lado del servidor.
  - Que devuelva a cada cliente el día y hora de conexión, y el tiempo que estuvo (o está) conectado.   
  <br>

- Realizar la misma aplicación tanto para C-S con Concurrencia Aparente (Select) como C-S Concurrente.

- De ser necesario agregar tiempo de espera, loop, sleep, con contadores para demorar los procesos.

- Implementar una limpieza de recuersos al salir del los programas (agregar opción de pregunta al usuario para cerrar los clientes).


## Servidor Concurrente

In [ ]:
import socket
import threading
import time
import struct
import datetime

clientes = []
mensajes = []
clientes_lock = threading.Lock()
mensajes_lock = threading.Lock()
apagado = threading.Event()

HOST = '127.0.0.1'
PORT = 6667

    
# ========================
# SERVIDOR TCP
# ========================
def proceso_hijo(conn, addr):
    inicio_cliente = time.perf_counter()
    cliente_id = f"{addr[0]}:{addr[1]}" #addr tiene tanto ip, como puerto. id:puerto

    with clientes_lock:
        clientes.append({
        "id": cliente_id,
        "inicio": inicio_cliente})

        print("Los clientes conectados son:")
        for c in clientes:
            conectado_hace = time.perf_counter() - c["inicio"]
            print(f"     {c['id']} - conectado hace {conectado_hace:.2f} s")

    try:
        conn.send("Servidor: Conectado con cliente\n".encode('utf-8'))
        while not apagado.is_set():
            file_size = conn.recv(1024)
            if file_size:
                tamano = struct.unpack(">Q",file_size)[0] #big endian, entero sin signo.
                print(f"Recibiendo archivo de: {addr}, tamano: {tamano} bytes")
                recibido =0
                nombre_arch = f"recibido_{addr[1]}.bin"
                with open(nombre_arch, "wb") as f:
                    while recibido < tamano:
                        restante = tamano- recibido
                        capacidad_lectura = min(1024, restante)
                
                        chunk = conn.recv(capacidad_lectura)
                        if not chunk:
                            break
            
                        f.write(chunk)
                        recibido += len(chunk)
                print(f"Archivo guardado exitosamente como {nombre_arch}")
                ahora = datetime.datetime.now()
                tiempo_conectado = time.perf_counter() - inicio_cliente
                mensaje = f"Date {ahora.strftime('%Y-%m-%d %H:%M')}, tiempo conectado: {tiempo_conectado:.2f} s\n"
                msg_bytes = mensaje.encode("utf-8")
                conn.sendall(struct.pack(">I", len(msg_bytes))) #envia la longitud del mensaje
                conn.sendall(msg_bytes)
            else:
                break
    except Exception as e:
        print(f"Error con {addr}: {e}")
    finally:
        conn.close()
        cliente_id = f"{addr[0]}:{addr[1]}"
        with clientes_lock:
            for c in clientes:
                if c["id"] == cliente_id:
                    clientes.remove(c)
                    break

def servidor_tcp():
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    sock.bind((HOST, PORT))
    sock.listen(5)
    sock.settimeout(1.0)
    print("Servidor TCP escuchando en", (HOST, PORT))
    try:
        while not apagado.is_set():
            try:
                conn, addr = sock.accept()
                threading.Thread(target=proceso_hijo, args=(conn, addr)).start()
            except socket.timeout:
                continue
    finally:
        sock.close()
        print("Servidor apagado.")

# ========================
# MAIN
# ========================
if __name__ == "__main__":
    # Inicia servidor TCP en un hilo
    threading.Thread(target=servidor_tcp, daemon=True).start()
    while not apagado.is_set():
        try:
            time.sleep(10)
            contador +=10
            print(f"Servidor encendido hace {contador} segundos")
        except KeyboardInterrupt:
            print("Cerrando servidor")
            apagado.set()

## Cliente concurrente

In [ ]:
from gc import enable
import socket
import os
import threading
import struct

HOST = '127.0.0.1'   # Cambiar IP si el servidor está en otra máquina
PORT = 6667          # Puerto del servidor

def enviar_archivo():
    file_name = input("Ingrese el nombre del archivo a enviar:")
    file_size = os.path.getsize(file_name)

    #lo mandas por chunks
    sock.sendall(file_size.to_bytes(8, byteorder='big'))

    with open(file_name,"rb") as f:
        while chunk := f.read(1024):
            sock.sendall(chunk)
            
    len_msg = struct.unpack(">I", sock.recv(4))[0]
    print(sock.recv(len_msg))
def escuchar():
    data = sock.recv(1024)
    if data:
        print("Servidor: ",data.decode('utf-8'))

if __name__ == "__main__":
    try:
        print("Creando socket...")
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        sock.settimeout(5)  # Tiempo máximo de espera para recibir datos

        print(f"Conectando a servidor en {HOST}:{PORT}...")
        sock.connect((HOST, PORT))
        threading.Thread(target=escuchar, daemon=True).start()
        print("Elija una opcion")
        print("(1)- Enviar \n (2)- Salir")
        op = input()
        while (op not in ['1','2']):
            op = input()
        while op!= 2:
            if((op==1)):
                threading.Thread(target=enviar_archivo, daemon=True).start()
            if(op==2):
                break 

    except ConnectionRefusedError:
        print("No se pudo conectar al servidor. Verifique si está activo.")
    except Exception as e:
        print(f"Ocurrió un error: {e}")
    finally:
        sock.close()
        print("Paso 6: Socket cerrado. Cliente finalizado.")


## Servidor Select

In [ ]:
# Socket_Servidor_Select.py
# Servidor con Select()

import datetime
import select
import socket
import sys
import queue
import struct
import os
import time
# Creando un socket TCP/IP 
servidor = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
servidor.setblocking(0)

# Hago Bind del socket al puerto
dir_servidor = ('localhost', 1000)
print('iniciando en {} port {}'.format(*dir_servidor),
      file=sys.stderr)
servidor.bind(dir_servidor)

# Escucho conexiones entrantes
servidor.listen(5)

# Sockect que espero leer. aka todos los sockets
entradas = [servidor]

# Sockets que espero enviar
salidas = []

# Cola de mensajes salientes
cola_mensajes = {}

# Diccionario de tiempos
tiempos_conectado = {}


try:
    while entradas:
        
        # Espero a que al menos uno de los sockets este listo para ser procesado

        print('esperando el próximo evento', file=sys.stderr)
        readable, writable, exceptional = select.select(entradas,
                                                        salidas,
                                                        entradas)

        if not (readable or writable or exceptional):
            print('  tiempo excedido....',
                file=sys.stderr)
            continue
        # Manejo entradas
        for s in readable:

            if s is servidor:
                # Un socket "leíble" está listo para aceptar conexiones
                con, dir_cliente = s.accept()
                print('  conexión desde: ', dir_cliente,
                    file=sys.stderr)
                con.setblocking(0)
                entradas.append(con)
                tiempos_conectado[con]=time.perf_counter()
                # Le asigno a la conexión una cola en la cuál quiero enviar
                cola_mensajes[con] = queue.Queue()

            else:
                # leer tamaño del nombre
                raw = s.recv(4)
                if raw:
                    name_len = struct.unpack(">I", raw)[0]
                    # leer nombre exacto
                    file_name = s.recv(name_len).decode()
                    
                    print('  Se espera recibir el archivo {!r} desde {}'.format(
                        file_name, s.getpeername()), file=sys.stderr,
                    )
                    file_name_to_create= f"{file_name}_{s.getpeername()}"
                    file_size=s.recv(8)
                    file_size = struct.unpack(">Q",file_size)[0] #big endian, entero sin signo.
                    recibido =0
                    with open(file_name_to_create, "wb") as f:
                        while recibido <file_size:
                            restante = file_size- recibido
                            capacidad_lectura = min(1024, restante)
                    
                            chunk = s.recv(capacidad_lectura)
                            if not chunk:
                                break
                
                            f.write(chunk)
                            recibido += len(chunk)
                    print(f"Archivo {file_name_to_create} guardado exitosamente")
                    
                    ahora = datetime.datetime.now()
                    tiempo = time.perf_counter() - tiempos_conectado[s]
                    respuesta=f"Se guardo exitosamente el archivo \n FECHA: {ahora.strftime('%Y-%m-%d %H:%M')}, tiempo conectado: {tiempo:.2f} s\n"

                    cola_mensajes[s].put(respuesta.encode())
                    # Agrego un canal de salida para la respuesta
                    if s not in salidas:
                        salidas.append(s)
                else:
                    # Si está vacío lo interpreto como una conexión a cerrar
                    print('  cerrando...', dir_cliente,
                        file=sys.stderr)
                    # dejo de escuchar en la conexión
                    if s in salidas:
                        salidas.remove(s)
                    entradas.remove(s)
                    del tiempos_conectado[con]
                    s.close()

                    # Rremueve mensaje de la cola
                    del cola_mensajes[s]

        # Administro salidas
        for s in writable:
            try:
                next_msg = cola_mensajes[s].get_nowait()
            except queue.Empty:
                # No hay mensaje en espera. Dejo de controlar para posibles escrituras

                print('  ', s.getpeername(), 'cola vacía',
                    file=sys.stderr)
                salidas.remove(s)
            else:
                print(' enviando {!r} a {}'.format(next_msg, s.getpeername()), file=sys.stderr)
                s.send(next_msg)

    # Administro condiciones excepcionales"
        for s in exceptional:
            print('excepción en', s.getpeername(),
                file=sys.stderr)
            # Dejo de escuchar en las conexiones
            entradas.remove(s)
            if s in salidas:
                salidas.remove(s)
            s.close()

            # Remuevo cola de mensajes
            del cola_mensajes[s]
except KeyboardInterrupt:
    print("Apagando servidor...")

finally:
    for s in entradas:
        s.close()
        entradas.remove(s)
        del cola_mensajes[s]

## Cliente Select

In [ ]:
from gc import enable
import socket
import os
import threading
import struct
import traceback

HOST = '127.0.0.1'   # Cambiar IP si el servidor está en otra máquina
PORT = 1000         # Puerto del servidor


def enviar_archivo():
    file_name = input("Ingrese el nombre del archivo a enviar:")
    file_size = os.path.getsize(file_name)
    #lo mandas por chunks
    sock.sendall(file_size.to_bytes(8, byteorder='big'))

    with open(file_name,"rb") as f:
        while chunk := f.read(1024):
            sock.sendall(chunk)

    len_msg = struct.unpack(">I", sock.recv(4))[0]
    print(sock.recv(len_msg))

if __name__ == "__main__":
    try:
        print("Creando socket...")
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        sock.settimeout(5)  # Tiempo máximo de espera para recibir datos

        print(f"Conectando a servidor en {HOST}:{PORT}...")
        sock.connect((HOST, PORT))
        op = 'Y'
        while (op=='Y'):
            print("Ingrese el nombre del archivo a enviar:")
            file_name = input()
            file_size = os.path.getsize(file_name)
            
            sock.sendall(struct.pack(">I", len(file_name)))
            sock.sendall(file_name.encode())

            sock.sendall(file_size.to_bytes(8, byteorder='big'))

            with open(file_name,"rb") as f:
                while chunk := f.read(1024):
                    sock.sendall(chunk)
            
            print(sock.recv(1024))
            op = input("Desea continuar enviando archivos? (Y/N)")
            while (op not in ['Y','N']):
                op = input()
        
    except ConnectionRefusedError:
        print("No se pudo conectar al servidor. Verifique si está activo.")
    except Exception as e:
        print(f"Ocurrió un error: {e}")
        traceback.print_exc()
    finally:
        sock.close()
        print("Paso 6: Socket cerrado. Cliente finalizado.")

